In [21]:
!pip install transformers==4.45.2 accelerate

In [22]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [23]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

In [24]:
tok = AutoTokenizer.from_pretrained(MODEL)

tok.pad_token = tok.eos_token
tok.padding_side = "left"

In [25]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="cuda"
)

In [26]:
print("Model loaded")

Model loaded


In [27]:
import time
import threading
from transformers import TextIteratorStreamer

In [28]:
def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600

    ids = tok(base + filler)["input_ids"][:n_tokens]

    return tok.decode(ids)

In [29]:
def measure_stream(prompt: str, new_tokens: int = 128):

    # تحويل النص إلى tokens
    enc = tok(prompt, return_tensors="pt").to("cuda")

    # تشغيل streaming
    streamer = TextIteratorStreamer(
        tok,
        skip_prompt=True,
        skip_special_tokens=True
    )

    kwargs = dict(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
        streamer=streamer
    )

    # تشغيل generate في thread منفصل
    th = threading.Thread(
        target=model.generate,
        kwargs=kwargs
    )

    # بداية الوقت
    t0 = time.time()

    th.start()

    stamps = []

    # كل مرة يصل token نسجل الوقت
    for _ in streamer:
        stamps.append(time.time())

    th.join()


    # TTFT = وقت أول token
    ttft = stamps[0] - t0


    # TPOT = متوسط الفرق بين tokens
    if len(stamps) > 1:
        gaps = [
            b-a for a,b in zip(stamps, stamps[1:])
        ]

        tpot = sum(gaps) / len(gaps)

    else:
        tpot = 0.0


    total = stamps[-1] - t0


    return {
        "ttft_s": round(ttft,4),
        "tpot_s": round(tpot,4),
        "total_s": round(total,4),
        "n_tokens": len(stamps)
    }

In [30]:
measure_stream(
    prompt_of_len(128),
    new_tokens=8
)

{'ttft_s': 0.0608, 'tpot_s': 0.0298, 'total_s': 0.2988, 'n_tokens': 9}

In [31]:
ttft_by_len = {}

for n in [128, 512, 2048]:

    r = measure_stream(prompt_of_len(n))

    ttft_by_len[str(n)] = r["ttft_s"]

    print(n, r)

128 {'ttft_s': 0.0498, 'tpot_s': 0.0472, 'total_s': 6.0916, 'n_tokens': 129}
512 {'ttft_s': 0.0848, 'tpot_s': 0.0668, 'total_s': 8.6353, 'n_tokens': 129}
2048 {'ttft_s': 0.3058, 'tpot_s': 0.0336, 'total_s': 4.6021, 'n_tokens': 129}


In [32]:
import gc

In [33]:
def kv_formula_kb_per_token(
    layers=28,
    kv_heads=2,
    head_dim=128,
    dbytes=2
):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024

In [34]:
formula = kv_formula_kb_per_token()

print("formula KB/token:", formula)

formula KB/token: 28.0


In [35]:
def cache_bytes(pkv):

    if hasattr(pkv, "key_cache"):

        tensors = list(pkv.key_cache) + list(pkv.value_cache)

    else:

        tensors = [
            t
            for layer in pkv
            for t in layer
        ]

    return sum(
        t.numel() * t.element_size()
        for t in tensors
    )

In [36]:
def measure_kv(context: int, new_tokens: int = 256):

    torch.cuda.empty_cache()
    gc.collect()

    torch.cuda.reset_peak_memory_stats()


    enc = tok(
        prompt_of_len(context),
        return_tensors="pt"
    ).to("cuda")


    before = torch.cuda.memory_allocated()


    out = model.generate(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
        use_cache=True,
        return_dict_in_generate=True
    )


    torch.cuda.synchronize()


    peak = torch.cuda.max_memory_allocated()


    total_tokens = out.sequences.shape[1]


    return {
        "context": context,

        "total_tokens": int(total_tokens),

        "peak_kb_per_token":
        round(
            (peak-before)
            / total_tokens
            /1024,
            1
        ),

        "kv_kb_per_token":
        round(
            cache_bytes(out.past_key_values)
            / total_tokens
            /1024,
            1
        )
    }

In [37]:
kv_rows = [
    measure_kv(c)
    for c in [512, 2048, 4096]
]


for r in kv_rows:
    print(r)

{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 63.4, 'kv_kb_per_token': 28.0}
{'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 84.0, 'kv_kb_per_token': 28.0}
{'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 88.0, 'kv_kb_per_token': 28.0}


In [38]:
import json

with open("kv_check.json","w") as f:

    json.dump(
        {
        "formula_kb_per_token": formula,
        "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
        "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"]
        },
        f
    )

In [40]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
QUEUE = [32, 32, 32, 256] * 6

In [42]:
import time

def static_queue(
    batch,
    prompt="Explain what an inference server does."
):

    t0 = time.time()

    useful = 0
    slots = 0


    for i in range(0, len(QUEUE), batch):

        # أخذ مجموعة بحجم batch
        chunk = QUEUE[i:i+batch]


        # أطول طلب يحدد وقت الباتش
        n = max(chunk)


        # تجهيز النصوص
        enc = tok(
            [prompt] * len(chunk),
            return_tensors="pt",
            padding=True
        ).to("cuda")


        # تشغيل التوليد
        model.generate(
            **enc,
            max_new_tokens=n,
            do_sample=False
        )


        # عدد التوكنات المفيدة
        useful += sum(chunk)


        # عدد الأماكن التي عالجتها GPU
        slots += n * len(chunk)



    dt = time.time() - t0


    return {

        "batch": batch,

        "wall_s": round(dt,2),

        "tokens_per_s":
            round(useful/dt,1),

        "slot_efficiency":
            round(useful/slots,3)

    }

In [43]:
batch_rows = {}

for n in [1,4,8]:

    r = static_queue(n)

    batch_rows[str(n)] = r["tokens_per_s"]

    print(r)

{'batch': 1, 'wall_s': 60.83, 'tokens_per_s': 34.7, 'slot_efficiency': 1.0}
{'batch': 4, 'wall_s': 42.07, 'tokens_per_s': 50.2, 'slot_efficiency': 0.344}
{'batch': 8, 'wall_s': 21.67, 'tokens_per_s': 97.5, 'slot_efficiency': 0.344}


In [44]:
import json

In [45]:
baselines = {

    "model": MODEL,

    "dtype": "fp16",

    "ttft_s": ttft_by_len,

    "tpot_s": measure_stream(
        prompt_of_len(512)
    )["tpot_s"],

    "batch": {
        k: v
        for k, v in batch_rows.items()
    }

}

In [46]:
with open("baselines.json", "w") as f:

    json.dump(
        baselines,
        f,
        indent=2
    )


print(
    json.dumps(
        baselines,
        indent=2
    )
)

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "dtype": "fp16",
  "ttft_s": {
    "128": 0.0498,
    "512": 0.0848,
    "2048": 0.3058
  },
  "tpot_s": 0.0329,
  "batch": {
    "1": 34.7,
    "4": 50.2,
    "8": 97.5
  }
}


In [47]:
from google.colab import files

files.download("baselines.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
import json

with open("baselines.json", "r") as f:
    baselines = json.load(f)

# تحقق من وجود البيانات المطلوبة
checks = []

checks.append("model" in baselines)
checks.append("dtype" in baselines)
checks.append("ttft_s" in baselines)
checks.append("tpot_s" in baselines)
checks.append("batch" in baselines)

# تحقق أن TTFT أقل من الوقت الكلي
checks.append(
    all(
        baselines["ttft_s"][k] > 0
        for k in baselines["ttft_s"]
    )
)

# تحقق أن batch 8 موجود
checks.append("8" in baselines["batch"])

if all(checks):
    print("GREEN CHECK: PASS")
else:
    print("GREEN CHECK: FAIL")

GREEN CHECK: PASS
